In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from rank_bm25 import BM25Okapi

# تنظیم مسیر ریشه پروژه (فرض بر این است که نوت‌بوک در پوشه notebooks/ است)
current_dir = Path.cwd()
if current_dir.name == "notebooks":
    root_dir = current_dir.parent
else:
    root_dir = current_dir

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

print(f"Project Root Directory: {root_dir}")

# ایمپورت ماژول نرمال‌سازی فارسی
try:
    from src.digikala_ai.utils.normalization import normalize_persian_text
    print("Normalization module imported successfully.")
except ImportError as e:
    print(f"Warning: Could not import normalization module directly ({e}). Using inline fallback if needed.")
    def normalize_persian_text(text):
        if not isinstance(text, str):
            return ""
        return text.strip()


Project Root Directory: d:\goal\programming\quera\boot_camp\project3\notebook
Normalization module imported successfully.


In [3]:
# بارگذاری دیتاست محصولات از مسیر استاندارد processed
processed_path = root_dir / "data" / "processed" / "products.parquet"

if not processed_path.exists():
    processed_path = Path(r"D:\goal\programming\quera\boot_camp\project3\data\processed\products.parquet")

print(f"Loading products from: {processed_path}")
products_df = pd.read_parquet(processed_path)

print(f"Total dataset shape: {products_df.shape}")
print("Columns available:", products_df.columns.tolist())


Loading products from: D:\goal\programming\quera\boot_camp\project3\data\processed\products.parquet


Total dataset shape: (1283496, 12)
Columns available: ['id', 'title_fa', 'Rate', 'Rate_cnt', 'Category1', 'Category2', 'Brand', 'Price', 'Seller', 'Is_Fake', 'min_price_last_month', 'sub_category']


In [4]:
# بررسی توزیع Category1 برای یافتن دسته‌هایی با تعداد محصول بالا و عناوین واضح
top_categories = products_df["Category1"].value_counts().head(15)
print("Top 15 Categories by Product Count:")
print(top_categories)


Top 15 Categories by Product Count:
Category1
اکسسوری زنانه و مردانه     95999
لباس زنانه                 93392
اسباب بازی                 92466
لباس مردانه                68178
اکسسوری زنانه              64470
اکسسوری مردانه             49383
دخترانه                    42931
دفتر و کاغذ و مقوا         42864
زیورآلات طلا زنانه         41098
کتاب چاپی غیر فارسی        39409
مراقبت پوست                29662
مواد غذایی                 28930
پسرانه                     27066
کتاب شعر و ادبیات          23345
برس‌ها و تجهیزات آرایشی    22749
Name: count, dtype: int64


In [5]:
# انتخاب دسته‌های واضح و غنی برای تست دقیق‌تر BM25
# (بر اساس ساختار کاتالوگ شما، دسته‌هایی مانند موبایل یا لوازم جانبی دیجیتال)
target_categories = ["موبایل", "لپ‌تاپ و اولترابوک", "هدفون، هدست و هندزفری"]

# فیلتر کردن دیتافریم برای داشتن یک pool بزرگ و باکیفیت
eval_pool_df = products_df[
    products_df["Category1"].isin(top_categories.head(5).index)
].dropna(subset=["title_fa", "id"]).copy()

print(f"Evaluation Pool Size: {len(eval_pool_df)}")
display(eval_pool_df[["id", "title_fa", "Category1", "Brand"]].head(5))


Evaluation Pool Size: 414505


,id,title_fa,Category1,Brand
46567,12635402,بازی فکری مدل دوز,اسباب بازی,متفرقه
46568,5110481,ست اسباب بازی ابزار مهندسی مدل R 04,اسباب بازی,متفرقه
46569,12106636,کارت آموزشی ریاضی انتشارات نارنجی مدل آموزش جد...,اسباب بازی,انتشارات نارنجی
46570,3351999,ملخ کواد کوپتر مدل SD,اسباب بازی,متفرقه
46571,12106639,کارت آموزشی ریاضی انتشارات نارنجی مدل آموزش جد...,اسباب بازی,انتشارات نارنجی


In [6]:
# استخراج و نرمال‌سازی عناوین برای ساخت کورپوس BM25
print("Normalizing corpus titles...")
eval_pool_df["normalized_title"] = eval_pool_df["title_fa"].apply(normalize_persian_text)
corpus_tokens = [title.split() for title in eval_pool_df["normalized_title"].tolist()]

print("Building BM25 index on evaluation pool...")
bm25_index = BM25Okapi(corpus_tokens)
print("BM25 index built successfully!")


Normalizing corpus titles...
Building BM25 index on evaluation pool...
BM25 index built successfully!


In [7]:
# انتخاب چند محصول واقعی به عنوان Ground Truth Target برای تست دقیق
# در اینجا ۳ محصول نمونه از پویایی دیتافریم انتخاب می‌کنیم
sample_targets = eval_pool_df.sample(n=3, random_state=42)

test_queries = []
for idx, row in sample_targets.iterrows():
    # ساخت کوئری از روی عنوان واقعی محصول (یا بخشی از آن)
    title = row["title_fa"]
    tokens = title.split()
    # ساخت کوئری ترکیبی از کلمات کلیدی عنوان
    query_str = " ".join(tokens[:4]) # ۴ کلمه اول عنوان
    test_queries.append({
        "target_id": row["id"],
        "original_title": title,
        "query": query_str,
        "category": row["Category1"]
    })

print(f"Generated {len(test_queries)} evaluation queries using Method A:")
for q in test_queries:
    print(f"- Target ID: {q['target_id']} | Category: {q['category']} | Query: {q['query']}")


Generated 3 evaluation queries using Method A:
- Target ID: 3265269 | Category: اسباب بازی | Query: پازل 150 تکه مدل
- Target ID: 9272831 | Category: لباس مردانه | Query: هودی ورزشی مردانه برساد
- Target ID: 11114946 | Category: لباس مردانه | Query: تی شرت آستین کوتاه


In [8]:
hit_at_1_list = []
hit_at_5_list = []
mrr_list = []

evaluation_results = []

for q in test_queries:
    target_id = q["target_id"]
    raw_q = q["query"]
    
    # 1. نرمال‌سازی کوئری
    norm_q = normalize_persian_text(raw_q)
    q_tokens = norm_q.split()
    
    # 2. دریافت امتیازات BM25 برای کل pool
    scores = bm25_index.get_scores(q_tokens)
    
    # 3. رتبه‌بندی نتایج
    temp_df = eval_pool_df.copy()
    temp_df["bm25_score"] = scores
    ranked = temp_df.sort_values(by="bm25_score", ascending=False).reset_index(drop=True)
    
    # 4. یافتن رتبه محصول هدف (Target ID)
    matched_indices = ranked.index[ranked["id"] == target_id].tolist()
    
    if matched_indices:
        rank = matched_indices[0] + 1 # رتبه از 1 شروع می‌شود
    else:
        rank = 999999 # اگر در pool نبود
        
    hit_1 = 1 if rank == 1 else 0
    hit_5 = 1 if rank <= 5 else 0
    rr = 1.0 / rank if rank <= len(ranked) else 0.0
    
    hit_at_1_list.append(hit_1)
    hit_at_5_list.append(hit_5)
    mrr_list.append(rr)
    
    evaluation_results.append({
        "Query": raw_q,
        "Target ID": target_id,
        "Target Rank": rank,
        "Hit@1": hit_1,
        "Hit@5": hit_5,
        "MRR": rr
    })

# نمایش گزارش نهایی ارزیابی
eval_summary_df = pd.DataFrame(evaluation_results)
print("=== BM25 Retrieval Evaluation Report (Day 2 M3) ===")
display(eval_summary_df.to_string(index=False))

print(f"\nMean Hit@1: {np.mean(hit_at_1_list):.4f}")
print(f"Mean Hit@5: {np.mean(hit_at_5_list):.4f}")
print(f"Mean MRR:   {np.mean(mrr_list):.4f}")


=== BM25 Retrieval Evaluation Report (Day 2 M3) ===


'                  Query  Target ID  Target Rank  Hit@1  Hit@5      MRR\n       پازل 150 تکه مدل    3265269           11      0      0 0.090909\nهودی ورزشی مردانه برساد    9272831            2      0      1 0.500000\n     تی شرت آستین کوتاه   11114946        12454      0      0 0.000080'


Mean Hit@1: 0.0000
Mean Hit@5: 0.3333
Mean MRR:   0.1970


In [9]:
# نمایش عنوان واقعی target و پنج نتیجهٔ اول برای هر query

for result in evaluation_results:
    query = result["Query"]
    target_id = result["Target ID"]
    target_rank = result["Target Rank"]

    print("=" * 100)
    print(f"Query: {query}")
    print(f"Target ID: {target_id}")
    print(f"Target Rank: {target_rank}")

    target_row = eval_pool_df[eval_pool_df["id"] == target_id]

    if len(target_row) == 0:
        print("Target product is not present in evaluation pool.")
        continue

    print("\nTarget product:")
    display(
        target_row[
            ["id", "title_fa", "Brand", "Category1"]
        ].to_string(index=False)
    )

    # اجرای دوبارهٔ query برای گرفتن رتبه‌بندی
    normalized_query = normalize_persian_text(query)
    query_tokens = normalized_query.split()

    scores = bm25_index.get_scores(query_tokens)

    ranked_debug = eval_pool_df.copy()
    ranked_debug["bm25_score"] = scores
    ranked_debug = ranked_debug.sort_values(
        by="bm25_score",
        ascending=False
    ).reset_index(drop=True)

    print("\nTop 5 retrieved products:")
    display(
        ranked_debug[
            ["id", "title_fa", "Brand", "Category1", "bm25_score"]
        ].head(5).to_string(index=False)
    )


Query: پازل 150 تکه مدل
Target ID: 3265269
Target Rank: 11

Target product:


'     id                      title_fa  Brand  Category1\n3265269 پازل 150 تکه مدل کیتی کد 0366 متفرقه اسباب بازی\n3265269 پازل 150 تکه مدل کیتی کد 0366 متفرقه اسباب بازی'


Top 5 retrieved products:


'      id                   title_fa  Brand  Category1  bm25_score\n12310278     پازل 150 تکه مدل بتمن  متفرقه اسباب بازی   19.266522\n10276045      پازل 150 تکه مدل 1001 متفرقه اسباب بازی   19.266522\n12310278     پازل 150 تکه مدل بتمن  متفرقه اسباب بازی   19.266522\n10275781 پازل 150 تکه مدل 1006\xa0چوبی متفرقه اسباب بازی   17.976942\n10275781 پازل 150 تکه مدل 1006\xa0چوبی متفرقه اسباب بازی   17.976942'

Query: هودی ورزشی مردانه برساد
Target ID: 9272831
Target Rank: 2

Target product:


'     id                                 title_fa Brand   Category1\n9272831 هودی ورزشی مردانه برساد مدل D109 رنگ سبز برساد لباس مردانه\n9272831 هودی ورزشی مردانه برساد مدل D109 رنگ سبز برساد لباس مردانه'

... (2 outputs left out)

Query: تی شرت آستین کوتاه
Target ID: 11114946
Target Rank: 12454

Target product:


'      id                                       title_fa  Brand   Category1\n11114946 تی شرت آستین کوتاه مردانه مدل EXIT NO کد b0172 متفرقه لباس مردانه'


Top 5 retrieved products:


'      id                                        title_fa  Brand   Category1  bm25_score\n10751865 تی شرت آستین کوتاه آستین کوتاه زنانه مدل دوربین متفرقه  لباس زنانه   10.893484\n11764167                      تی شرت آستین کوتاه  کد 149 متفرقه  لباس زنانه   10.587967\n11245705                  تی شرت آستین کوتاه مردانه\xa0کد84 متفرقه لباس مردانه   10.587967\n10150435                    تی شرت آستین کوتاه\xa0کد TI-001 متفرقه  لباس زنانه   10.587967\n11247311                        تی شرت آستین کوتاه کد 96 متفرقه  لباس زنانه   10.587967'

In [10]:
# سلول اصلاحی: Deduplication دقیق محصولات بر اساس ID
print(f"Original products_df shape: {products_df.shape}")

# حذف رکوردهای کاملاً تکراری و یکتاسازی بر اساس id محصول
products_clean_df = products_df.drop_duplicates(subset=["id"]).copy()

print(f"Cleaned products_df shape (Unique IDs): {products_clean_df.shape}")

# بررسی اینکه آیا هنوز ID تکراری داریم؟
print(f"Number of duplicate IDs remaining: {products_clean_df['id'].duplicated().sum()}")


Original products_df shape: (1283496, 12)
Cleaned products_df shape (Unique IDs): (948352, 12)
Number of duplicate IDs remaining: 0


In [11]:
# بررسی IDهایی که در دادهٔ اصلی بیش از یک ردیف داشته‌اند

duplicate_id_mask = products_df["id"].duplicated(keep=False)

duplicate_ids_df = products_df.loc[
    duplicate_id_mask,
    ["id", "title_fa", "Category1", "Brand"]
].sort_values(["id", "title_fa"])

print(f"Rows belonging to duplicated IDs: {len(duplicate_ids_df):,}")
print(f"Number of duplicated IDs: {duplicate_ids_df['id'].nunique():,}")

display(duplicate_ids_df.head(20))


Rows belonging to duplicated IDs: 593,862
Number of duplicated IDs: 258,718


,id,title_fa,Category1,Brand
344968,12298,ساعت مچی عقربه ای مردانه کاسیو جی شاک GA-110-1ADR,اکسسوری مردانه,کاسیو
350656,12298,ساعت مچی عقربه ای مردانه کاسیو جی شاک GA-110-1ADR,اکسسوری مردانه,کاسیو
1057440,19287,آلبوم موسیقی آتشی در نیستان اثر شهرام ناظری,موسیقی با کلام,چهار باغ
1057447,19287,آلبوم موسیقی آتشی در نیستان اثر شهرام ناظری,موسیقی با کلام,چهار باغ
1056734,22572,آلبوم موسیقی عشق است اثر ناصر عبداللهی و امیر ...,موسیقی با کلام,دارینوش
1056766,22572,آلبوم موسیقی عشق است اثر ناصر عبداللهی و امیر ...,موسیقی با کلام,دارینوش
344702,26147,ساعت مچی عقربه ای مردانه کاسیو کاسیو-GA-100A-9A,اکسسوری مردانه,کاسیو
350567,26147,ساعت مچی عقربه ای مردانه کاسیو کاسیو-GA-100A-9A,اکسسوری مردانه,کاسیو
320417,27925,ساعت مچی عقربه ای مردانه فستینا مدل F16600/5,اکسسوری مردانه,فستینا
352360,27925,ساعت مچی عقربه ای مردانه فستینا مدل F16600/5,اکسسوری مردانه,فستینا


In [12]:
# بررسی اینکه IDهای تکراری به اطلاعات متفاوتی اشاره می‌کنند یا نه

duplicate_id_mask = products_df["id"].duplicated(keep=False)

duplicate_id_groups = (
    products_df.loc[duplicate_id_mask]
    .groupby("id")
    .agg(
        row_count=("id", "size"),
        title_count=("title_fa", "nunique"),
        category_count=("Category1", "nunique"),
        brand_count=("Brand", "nunique"),
    )
)

print("تعداد IDهای تکراری:", len(duplicate_id_groups))
print(
    "IDهای دارای بیش از یک عنوان:",
    int((duplicate_id_groups["title_count"] > 1).sum())
)
print(
    "IDهای دارای بیش از یک دسته:",
    int((duplicate_id_groups["category_count"] > 1).sum())
)
print(
    "IDهای دارای بیش از یک برند:",
    int((duplicate_id_groups["brand_count"] > 1).sum())
)

# نمایش چند مورد ناسازگار، اگر وجود داشته باشد
conflicting_ids = duplicate_id_groups[
    (duplicate_id_groups["title_count"] > 1)
    | (duplicate_id_groups["category_count"] > 1)
    | (duplicate_id_groups["brand_count"] > 1)
].index

print("تعداد IDهای دارای اطلاعات متناقض:", len(conflicting_ids))

if len(conflicting_ids) > 0:
    display(
        products_df.loc[
            products_df["id"].isin(conflicting_ids),
            ["id", "title_fa", "Category1", "Brand"]
        ]
        .sort_values("id")
        .head(30)
    )


تعداد IDهای تکراری: 258718
IDهای دارای بیش از یک عنوان: 6
IDهای دارای بیش از یک دسته: 0
IDهای دارای بیش از یک برند: 2
تعداد IDهای دارای اطلاعات متناقض: 8


,id,title_fa,Category1,Brand
1262460,4262387,کفش پیاده روی مردانه مدل ر,کفش مردانه,متفرقه
1270153,4262387,کفش پیاده روی مردانه مدل Sk005,کفش مردانه,متفرقه
984841,6079756,سرم ضد چروک اولاین مدل Sos Instant Lifting حجم...,مراقبت پوست,اولاین
1008339,6079756,سرم ضد چروک اولاین مدل Sos Instant Lifting حجم...,مراقبت پوست,متفرقه
1250680,9590146,واکس براق کننده کفش پی واکس مدل 65g به همراه ا...,کفش زنانه و مردانه,متفرقه
1250697,9590146,واکس براق کننده کفش پی واکس مدل 65g به همراه ا...,کفش زنانه و مردانه,متفرقه
1241672,9590146,واکس براق کننده کفش پی واکس مدل 65g به همراه ا...,کفش زنانه و مردانه,پی واکس
987787,11587282,تونر ویتالیر مدل Activit حجم 200 میلی لیتر,مراقبت پوست,ویتالیر
992945,11587282,تونر پاک کننده صورت ویتالیر مدل Activit حجم 20...,مراقبت پوست,ویتالیر
1005133,11587282,تونر پاک کننده صورت ویتالیر مدل Activit حجم 20...,مراقبت پوست,ویتالیر


In [13]:
# نمایش تمام ردیف‌های مربوط به IDهای دارای عنوان یا برند متفاوت

conflicting_ids = duplicate_id_groups[
    (duplicate_id_groups["title_count"] > 1)
    | (duplicate_id_groups["category_count"] > 1)
    | (duplicate_id_groups["brand_count"] > 1)
].index

conflicting_rows = (
    products_df.loc[
        products_df["id"].isin(conflicting_ids),
        ["id", "title_fa", "Category1", "Brand", "Price", "Rate", "Rate_cnt"]
    ]
    .sort_values(["id", "title_fa"])
)

print(f"Conflicting IDs: {len(conflicting_ids)}")
print(f"Rows to inspect: {len(conflicting_rows)}")

display(conflicting_rows.to_string(index=False))


Conflicting IDs: 8
Rows to inspect: 20


'      id                                                                              title_fa              Category1    Brand    Price  Rate  Rate_cnt\n 4262387                                                        کفش پیاده روی مردانه مدل Sk005             کفش مردانه   متفرقه  2180000    82      4289\n 4262387                                                            کفش پیاده روی مردانه مدل ر             کفش مردانه   متفرقه  2120000    82      4280\n 6079756                           سرم ضد چروک اولاین مدل Sos Instant Lifting حجم 18 میلی لیتر            مراقبت پوست   اولاین  4900000    80         1\n 6079756                           سرم ضد چروک اولاین مدل Sos Instant Lifting حجم 18 میلی لیتر            مراقبت پوست   متفرقه  4900000    80         1\n 9590146                        واکس براق کننده کفش پی واکس مدل 65g به همراه اسفنج\xa0بسته 6 عددی     کفش زنانه و مردانه  پی واکس  3990000    88         9\n 9590146                        واکس براق کننده کفش پی واکس مدل 65g به همراه ا

In [14]:
print("Shape:", products_clean_df.shape)
print("Duplicate IDs:", products_clean_df["id"].duplicated().sum())
print("Missing titles:", products_clean_df["title_fa"].isna().sum())

display(
    products_clean_df[["id", "title_fa", "Category1"]].head()
)


Shape: (948352, 12)
Duplicate IDs: 0
Missing titles: 0


,id,title_fa,Category1
0,7096438,آبسلانگ مدل s5 بسته 250 عددی,آبسلانگ
1,2845119,آبسلانگ مدل M-1 بسته 400 عددی,آبسلانگ
2,6117745,آبسلانگ مدل m50 مجموعه 500 عددی,آبسلانگ
3,1912926,استند ابسلانگ مدل S01,آبسلانگ
4,6335462,آبسلانگ نوری تسلامد مدل All-in-One,آبسلانگ


In [15]:
import re

# تست و اعمال نرمال‌سازی روی عنوان‌ها (برای ستون نمایشی اصلی حفظ می‌شود)
# یک ستون جدید برای متن نرمال‌شده می‌سازیم که در BM25 استفاده شود:

print("در حال اعمال نرمال‌سازی روی عنوان محصولات...")
products_clean_df["normalized_title"] = products_clean_df["title_fa"].apply(normalize_persian_text)

# نمایش چند نمونه برای بررسی کیفیت نرمال‌سازی
display(
    products_clean_df[["id", "title_fa", "normalized_title"]].head(3)
)


در حال اعمال نرمال‌سازی روی عنوان محصولات...


,id,title_fa,normalized_title
0,7096438,آبسلانگ مدل s5 بسته 250 عددی,آبسلانگ مدل s5 بسته 250 عددی
1,2845119,آبسلانگ مدل M-1 بسته 400 عددی,آبسلانگ مدل M-1 بسته 400 عددی
2,6117745,آبسلانگ مدل m50 مجموعه 500 عددی,آبسلانگ مدل m50 مجموعه 500 عددی


In [16]:
import time
from rank_bm25 import BM25Okapi

print("در حال توکنایز کردن عناوین برای BM25...")
t0 = time.time()

# ساده‌ترین و استانداردترین توکنایزر بر اساس فاصله‌ها
tokenized_corpus = [doc.split(" ") for doc in products_clean_df["normalized_title"]]

print(f"توکنایز کردن تکمیل شد در {time.time() - t0:.2f} ثانیه.")

print("در حال ساخت ایندکس BM25...")
t0 = time.time()
bm25 = BM25Okapi(tokenized_corpus)
print(f"ساخت ایندکس BM25 تکمیل شد در {time.time() - t0:.2f} ثانیه.")


در حال توکنایز کردن عناوین برای BM25...
توکنایز کردن تکمیل شد در 11.14 ثانیه.
در حال ساخت ایندکس BM25...
ساخت ایندکس BM25 تکمیل شد در 13.38 ثانیه.


In [17]:
import numpy as np

def search_bm25(query_text, top_k=5):
    # نرمال‌سازی و توکنایز کردن کوئری ورودی با همان سیاست
    normalized_q = normalize_persian_text(query_text)
    tokenized_query = normalized_q.split(" ")
    
    # گرفتن امتیازات از ایندکس BM25
    scores = bm25.get_scores(tokenized_query)
    
    # استخراج برترین نتایج
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "id": products_clean_df.iloc[idx]["id"],
            "title_fa": products_clean_df.iloc[idx]["title_fa"],
            "Category1": products_clean_df.iloc[idx]["Category1"],
            "bm25_score": scores[idx]
        })
    return results

# تست سریع جستجو با یک کوئری نمونه از کاتالوگ
sample_query = "آبسلانگ مدل s5"
print(f"نتایج جستجو برای کوئری: '{sample_query}'\n")
for res in search_bm25(sample_query, top_k=3):
    print(f"ID: {res['id']} | Score: {res['bm25_score']:.2f} | Title: {res['title_fa']}")


نتایج جستجو برای کوئری: 'آبسلانگ مدل s5'

ID: 7096438 | Score: 27.54 | Title: آبسلانگ مدل s5 بسته 250 عددی
ID: 8706808 | Score: 19.04 | Title: آویزعروسکی نیکی مدل s5
ID: 11742933 | Score: 16.69 | Title: شلوار زنانه مدل s5 بغل چاکدار


In [18]:
# تعریف مجموعه‌ای از کوئری‌های آزمایشی روش الف (Method A) به همراه شناسه هدف واقعی
evaluation_test_cases = [
    {"query": "آبسلانگ مدل s5", "target_id": 7096438},
    {"query": "تبلت سامسونگ A7 Lite", "target_id": 5549287},
    {"query": "پازل 150 تکه مدل", "target_id": 3265269},
    {"query": "هودی ورزشی مردانه برساد", "target_id": 9272831},
    {"query": "تی شرت آستین کوتاه", "target_id": 11114946},
]

def evaluate_bm25_set(test_cases):
    hits_at_1 = []
    hits_at_5 = []
    mrr_list = []
    
    report_rows = []
    
    for case in test_cases:
        q = case["query"]
        target_id = case["target_id"]
        
        # جستجو تا رتبه 20 برای یافتن رتبه دقیق هدف
        results = search_bm25(q, top_k=20)
        
        rank = None
        for i, res in enumerate(results):
            if res["id"] == target_id:
                rank = i + 1  # 1-based rank
                break
                
        hit_1 = 1 if rank == 1 else 0
        hit_5 = 1 if rank is not None and rank <= 5 else 0
        mrr = (1.0 / rank) if rank is not None and rank <= 20 else 0.0
        
        hits_at_1.append(hit_1)
        hits_at_5.append(hit_5)
        mrr_list.append(mrr)
        
        report_rows.append({
            "Query": q,
            "Target ID": target_id,
            "Target Rank": rank if rank else ">20",
            "Hit@1": hit_1,
            "Hit@5": hit_5,
            "MRR": f"{mrr:.4f}"
        })
        
    import pandas as pd
    eval_df = pd.DataFrame(report_rows)
    print(f"=== BM25 Retrieval Evaluation Report (Day 2 M3) ===")
    print(f"Mean Hit@1: {np.mean(hits_at_1):.4f}")
    print(f"Mean Hit@5: {np.mean(hits_at_5):.4f}")
    print(f"Mean MRR:   {np.mean(mrr_list):.4f}\n")
    return eval_df

eval_report_df = evaluate_bm25_set(evaluation_test_cases)
display(eval_report_df)


=== BM25 Retrieval Evaluation Report (Day 2 M3) ===
Mean Hit@1: 0.6000
Mean Hit@5: 0.6000
Mean MRR:   0.6286



,Query,Target ID,Target Rank,Hit@1,Hit@5,MRR
0,آبسلانگ مدل s5,7096438,1,1,1,1.0000
1,تبلت سامسونگ A7 Lite,5549287,1,1,1,1.0000
2,پازل 150 تکه مدل,3265269,7,0,0,0.1429
3,هودی ورزشی مردانه برساد,9272831,1,1,1,1.0000
4,تی شرت آستین کوتاه,11114946,>20,0,0,0.0000
